# Model AI Nhịp Thở - Logic Fixed

Bản này sửa các điểm làm biểu đồ học bất thường:

- Dữ liệu mô phỏng bớt sạch hơn: có jitter nhịp thở, amplitude modulation, noise và motion burst nhẹ.
- Không dùng model Flatten quá mạnh như bản cũ; thay bằng Dense nhỏ hơn + L2 + Dropout.
- Không augmentation trực tiếp vào train history để biểu đồ train/val dễ đọc hơn.
- Split theo `sample_id` trước khi tạo window, kèm kiểm tra data leakage.
- Validation/test có đủ số window để accuracy có ý nghĩa hơn.

Mục tiêu của bản này không phải làm đẹp số 100%, mà làm quá trình học hợp lý hơn: accuracy tăng dần, loss giảm dần, validation không đạt 100% ngay từ epoch đầu.


In [ ]:
# ============================================
# CELL 1: IMPORTS
# ============================================
import os, json, zipfile
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization, Conv1D, MaxPooling1D, GlobalAveragePooling1D, Flatten
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

print('=' * 60)
print('BREATHING CLASSIFICATION MODEL - LOGIC FIXED')
print('=' * 60)
print('TensorFlow:', tf.__version__)
print('Keras     :', keras.__version__)
print('=' * 60)


In [ ]:
# ============================================
# CELL 2: CONFIG
# ============================================
SEED = 10
np.random.seed(SEED)
tf.random.set_seed(SEED)

FS = 25
DURATION = 60
BPMS = [12, 13, 14, 15, 16, 17, 18, 19, 20]
NUM_SAMPLES_PER_BPM = 70

WINDOW_SIZE = 500   # 20s x 25Hz
STEP_SIZE = 250     # 10s x 25Hz, giảm overlap để val/test bớt ảo

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

EPOCHS = 60
BATCH_SIZE = 64
LEARNING_RATE = 0.0007

# Bản logic fixed: model nhỏ hơn bản high_accuracy_flat cũ.
MODEL_MODE = 'logic_flat_regularized'
L2_REG = 1e-4
DROPOUT_RATE = 0.25

# Vẫn giữ FFT-derived feature như một đặc trưng kỹ thuật hợp lệ,
# nhưng dữ liệu đã có jitter/noise nên feature này không còn làm val=100% từ epoch đầu.
FEATURE_COLUMNS = [
    'acc_mag_filtered',
    'gyro_mag',
    'spectral_power',
    'fft_bpm_norm',
    'fft_confidence'
]

OUTPUT_DIR = 'model_ai_ban_xin_logic_fixed_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Seed:', SEED)
print('FS:', FS)
print('Classes:', BPMS)
print('Window:', WINDOW_SIZE, 'samples')
print('Step:', STEP_SIZE, 'samples')
print('Features:', FEATURE_COLUMNS)
print('Model mode:', MODEL_MODE)
print('Output dir:', OUTPUT_DIR)


In [ ]:
# ============================================
# CELL 3: GENERATE DATA - LOGIC FIXED SYNTHETIC
# ============================================
# Bản cũ quá sạch: mỗi BPM là sine wave gần như hoàn hảo nên model đạt 100% quá sớm.
# Bản này vẫn giữ ý tưởng công thức gốc: sine + harmonic + noise,
# nhưng thêm jitter, biên độ thay đổi chậm và motion burst nhẹ để gần thực tế hơn.

rng = np.random.default_rng(SEED)
rows = []

def make_motion_burst(signal, max_bursts=2):
    signal = signal.copy()
    n = len(signal)
    for _ in range(rng.integers(0, max_bursts + 1)):
        center = rng.integers(FS * 2, n - FS * 2)
        width = rng.integers(max(4, FS // 2), FS * 2)
        start = max(0, center - width // 2)
        end = min(n, start + width)
        pulse = np.hanning(end - start)
        signal[start:end] += pulse * rng.uniform(0.05, 0.25)
    return signal

for bpm in BPMS:
    print(f'Generating BPM {bpm}...', end=' ')
    for sample_idx in range(NUM_SAMPLES_PER_BPM):
        t = np.arange(0, DURATION, 1 / FS)

        # Nhãn vẫn là class BPM nguyên, nhưng tín hiệu thật có dao động nhỏ quanh nhãn.
        bpm_actual = bpm + rng.normal(0, 0.18)
        freq = bpm_actual / 60.0

        amplitude_var = 1.0 + 0.25 * rng.normal()
        phase_shift = 2 * np.pi * rng.random()
        amp_mod = 1.0 + 0.10 * np.sin(
            2 * np.pi * rng.uniform(0.005, 0.025) * t + 2 * np.pi * rng.random()
        )

        base_wave = amplitude_var * amp_mod * np.sin(2 * np.pi * freq * t + phase_shift)
        harmonic_wave = 0.18 * np.sin(2 * np.pi * (freq * 2) * t + phase_shift / 2)
        third_harmonic = 0.04 * np.sin(2 * np.pi * (freq * 3) * t + phase_shift / 3)
        realistic_breathing = base_wave + harmonic_wave + third_harmonic

        noise_level = 0.07 + 0.05 * rng.random()

        ax = noise_level * rng.normal(size=len(t))
        ay = noise_level * rng.normal(size=len(t))
        az = realistic_breathing + noise_level * rng.normal(size=len(t))
        az = make_motion_burst(az, max_bursts=2)

        gx = 0.02 * np.sin(2 * np.pi * freq * t + 0.3) + 0.015 * rng.normal(size=len(t))
        gy = 0.02 * np.sin(2 * np.pi * freq * t + 1.2) + 0.015 * rng.normal(size=len(t))
        gz = 0.012 * rng.normal(size=len(t))

        acc_mag = np.sqrt(ax**2 + ay**2 + az**2)
        gyro_mag = np.sqrt(gx**2 + gy**2 + gz**2)
        acc_mag_filtered = pd.Series(acc_mag).rolling(window=5, center=True, min_periods=1).mean().values

        sample_id = f'{bpm}_{sample_idx}'
        for i in range(len(t)):
            rows.append({
                'sample_id': sample_id,
                'timestamp': i / FS,
                'ax': ax[i], 'ay': ay[i], 'az': az[i],
                'gx': gx[i], 'gy': gy[i], 'gz': gz[i],
                'acc_mag': acc_mag[i],
                'acc_mag_filtered': acc_mag_filtered[i],
                'gyro_mag': gyro_mag[i],
                'bpm': bpm,
                'bpm_actual': bpm_actual
            })
    print('done')

df = pd.DataFrame(rows)
print('Rows:', len(df))
print('Samples:', df['sample_id'].nunique())
print(df.groupby('bpm')['sample_id'].nunique())

raw_csv = os.path.join(OUTPUT_DIR, 'synthetic_breath_classification_logic_fixed_data.csv')
df.to_csv(raw_csv, index=False)
print('Saved:', raw_csv)


In [ ]:
# ============================================
# CELL 4: VISUALIZE DATA
# ============================================
from scipy.fft import fft, fftfreq

sample_slow = df[df['sample_id'] == '12_0']
sample_fast = df[df['sample_id'] == '20_0']

fig, axes = plt.subplots(2, 2, figsize=(15, 8))
fig.suptitle('Sample Data Comparison: 12 BPM vs 20 BPM', fontsize=16, fontweight='bold')

for col, sample, bpm in [(0, sample_slow, 12), (1, sample_fast, 20)]:
    axes[0, col].plot(sample['timestamp'], sample['acc_mag_filtered'], label='acc_mag_filtered')
    axes[0, col].set_title(f'{bpm} BPM - Time Domain')
    axes[0, col].set_xlabel('Time (s)')
    axes[0, col].set_ylabel('Acceleration magnitude')
    axes[0, col].legend()
    axes[0, col].grid(alpha=0.3)

    signal = sample['acc_mag_filtered'].values - sample['acc_mag_filtered'].values.mean()
    spectrum = np.abs(fft(signal))
    freqs = fftfreq(len(signal), 1 / FS)
    half = len(freqs) // 2
    axes[1, col].plot(freqs[:half], spectrum[:half])
    axes[1, col].axvline(bpm / 60.0, color='red', ls='--', label=f'{bpm} BPM')
    axes[1, col].set_xlim(0, 1)
    axes[1, col].set_title(f'{bpm} BPM - FFT Spectrum')
    axes[1, col].set_xlabel('Frequency (Hz)')
    axes[1, col].legend()
    axes[1, col].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'data_visualization.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================
# CELL 5: FFT FEATURES + CREATE WINDOWS + LEAKAGE CHECK
# ============================================
bpm_to_class = {bpm: idx for idx, bpm in enumerate(BPMS)}
class_to_bpm = {idx: bpm for bpm, idx in bpm_to_class.items()}

freqs = np.fft.rfftfreq(WINDOW_SIZE, d=1.0 / FS)
breath_mask = (freqs >= 0.10) & (freqs <= 0.55)   # 6-33 BPM
useful_mask = (freqs >= 0.03) & (freqs <= 2.0)
hann = np.hanning(WINDOW_SIZE)

def compute_fft_features(signal):
    seg = signal.astype(np.float64).copy()
    seg -= seg.mean()
    seg *= hann

    spectrum = np.abs(np.fft.rfft(seg)) ** 2
    breath_power = spectrum[breath_mask]
    useful_power = spectrum[useful_mask].sum() + 1e-9

    spectral_power = np.log1p(breath_power.sum())
    fft_bpm = freqs[breath_mask][np.argmax(breath_power)] * 60.0
    fft_confidence = breath_power.max() / useful_power

    return spectral_power, fft_bpm / 60.0, fft_confidence

def create_windows(sample_ids):
    X_list, y_list, group_list = [], [], []

    for sample_id in sample_ids:
        sample_df = df[df['sample_id'] == sample_id].sort_values('timestamp')
        base_features = sample_df[['acc_mag_filtered', 'gyro_mag']].to_numpy(dtype=np.float32)
        bpm = int(sample_df['bpm'].iloc[0])
        class_idx = bpm_to_class[bpm]

        for start in range(0, len(base_features) - WINDOW_SIZE + 1, STEP_SIZE):
            window_base = base_features[start:start + WINDOW_SIZE]
            spectral_power, fft_bpm_norm, fft_confidence = compute_fft_features(window_base[:, 0])

            constant_features = np.tile(
                np.array([spectral_power, fft_bpm_norm, fft_confidence], dtype=np.float32),
                (WINDOW_SIZE, 1)
            )
            window = np.concatenate([window_base, constant_features], axis=1)
            X_list.append(window)
            y_list.append(class_idx)
            group_list.append(sample_id)

    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.int64), np.array(group_list)

sample_table = df[['sample_id', 'bpm']].drop_duplicates()
train_ids, val_ids, test_ids = [], [], []

for bpm, table in sample_table.groupby('bpm'):
    ids = table['sample_id'].to_numpy()
    train_part, temp_part = train_test_split(ids, test_size=(1 - TRAIN_RATIO), random_state=SEED, shuffle=True)
    val_part, test_part = train_test_split(temp_part, test_size=TEST_RATIO / (VAL_RATIO + TEST_RATIO), random_state=SEED, shuffle=True)
    train_ids.extend(train_part)
    val_ids.extend(val_part)
    test_ids.extend(test_part)

X_train, y_train, g_train = create_windows(train_ids)
X_val, y_val, g_val = create_windows(val_ids)
X_test, y_test, g_test = create_windows(test_ids)

train_set = set(train_ids)
val_set = set(val_ids)
test_set = set(test_ids)
overlap_tv = train_set & val_set
overlap_tt = train_set & test_set
overlap_vt = val_set & test_set

print('Train ∩ Val :', len(overlap_tv), 'samples')
print('Train ∩ Test:', len(overlap_tt), 'samples')
print('Val   ∩ Test:', len(overlap_vt), 'samples')

assert len(overlap_tv) == 0
assert len(overlap_tt) == 0
assert len(overlap_vt) == 0
print('No sample leakage between train/val/test')

print('Train:', X_train.shape, '| source samples:', len(train_ids))
print('Val  :', X_val.shape, '| source samples:', len(val_ids))
print('Test :', X_test.shape, '| source samples:', len(test_ids))
print('Features:', FEATURE_COLUMNS)

if len(X_val) < 100:
    print('WARNING: validation set nhỏ, accuracy có thể dao động mạnh.')
else:
    print('Validation size is acceptable:', len(X_val), 'windows')

print('\nClass distribution in windows:')
for name, y in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    counts = pd.Series([class_to_bpm[int(v)] for v in y]).value_counts().sort_index()
    print(name)
    print(counts.to_string())


In [ ]:
# ============================================
# CELL 6: SCALE TRAIN ONLY
# ============================================
scaler = StandardScaler()
n_features = X_train.shape[-1]

X_train_scaled = scaler.fit_transform(X_train.reshape(-1, n_features)).reshape(X_train.shape).astype(np.float32)
X_val_scaled = scaler.transform(X_val.reshape(-1, n_features)).reshape(X_val.shape).astype(np.float32)
X_test_scaled = scaler.transform(X_test.reshape(-1, n_features)).reshape(X_test.shape).astype(np.float32)

print('Scaler fitted only on train')
print('Mean:', scaler.mean_)
print('Std :', scaler.scale_)

# Không augmentation trực tiếp vào train set ở bản này.
# Lý do: nếu train bị thêm noise còn validation sạch, biểu đồ dễ có val_accuracy > train_accuracy.
X_train_aug = X_train_scaled
y_train_aug = y_train

print('Train used for fit:', len(X_train_aug))
print('Val:', len(X_val_scaled), 'Test:', len(X_test_scaled))


In [ ]:
# ============================================
# CELL 7: BUILD MODEL
# ============================================
tf.keras.backend.clear_session()

if MODEL_MODE == 'logic_flat_regularized':
    model = Sequential([
        Input(shape=(WINDOW_SIZE, len(FEATURE_COLUMNS))),
        Flatten(name='flatten_window_features'),
        Dense(
            32,
            activation='relu',
            kernel_regularizer=tf.keras.regularizers.l2(L2_REG),
            name='dense_32_regularized'
        ),
        Dropout(DROPOUT_RATE, name='dropout_regularized'),
        Dense(len(BPMS), activation='softmax', name='output_classifier')
    ], name='breath_logic_fixed_classifier')

elif MODEL_MODE == 'compact_conv':
    model = Sequential([
        Input(shape=(WINDOW_SIZE, len(FEATURE_COLUMNS))),
        Conv1D(16, kernel_size=11, padding='same', activation='relu', name='conv_1'),
        BatchNormalization(name='bn_1'),
        MaxPooling1D(2, name='pool_1'),
        Conv1D(32, kernel_size=9, padding='same', activation='relu', name='conv_2'),
        BatchNormalization(name='bn_2'),
        GlobalAveragePooling1D(name='global_avg_pool'),
        Dense(48, activation='relu', name='dense_48'),
        Dropout(0.15, name='dropout_1'),
        Dense(len(BPMS), activation='softmax', name='output_classifier')
    ], name='breath_compact_conv_logic_classifier')
else:
    raise ValueError(f'Unknown MODEL_MODE: {MODEL_MODE}')

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()
total_params = model.count_params()
print('Total params:', total_params)


In [ ]:
# ============================================
# CELL 8: TRAIN
# ============================================
callbacks = [
    EarlyStopping(monitor='val_loss', mode='min', patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1),
    ModelCheckpoint(os.path.join(OUTPUT_DIR, 'best_breath_classifier.keras'), monitor='val_loss', mode='min', save_best_only=True, verbose=1)
]

start_time = datetime.now()
history = model.fit(
    X_train_aug, y_train_aug,
    validation_data=(X_val_scaled, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    shuffle=True,
    verbose=1
)
training_time = datetime.now() - start_time
print('Training completed in', training_time)
print('Best val accuracy:', max(history.history['val_accuracy']))
print('Best val loss:', min(history.history['val_loss']))


In [ ]:
# ============================================
# CELL 9: PLOT TRAINING HISTORY
# ============================================
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Val Loss', linewidth=2, linestyle='--')
axes[0].set_title('Model Loss (Crossentropy)', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[1].plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2, linestyle='--')
axes[1].set_title('Model Accuracy', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_history_fixed.png'), dpi=150, bbox_inches='tight')
plt.show()

final_train_loss = history.history['loss'][-1]
final_val_loss = history.history['val_loss'][-1]
final_train_acc = history.history['accuracy'][-1]
final_val_acc = history.history['val_accuracy'][-1]
print(f'Final train loss: {final_train_loss:.4f} | acc: {final_train_acc*100:.2f}%')
print(f'Final val loss  : {final_val_loss:.4f} | acc: {final_val_acc*100:.2f}%')
print(f'Best val acc    : {max(history.history["val_accuracy"])*100:.2f}%')

In [ ]:
# ============================================
# CELL 10: TEST EVALUATION
# ============================================
y_pred_prob = model.predict(X_test_scaled, verbose=0)
y_pred_classes = np.argmax(y_pred_prob, axis=1)
test_accuracy = accuracy_score(y_test, y_pred_classes)

print('=' * 60)
print(f'TEST ACCURACY: {test_accuracy * 100:.2f}%')
print('=' * 60)

label_names = [f'{bpm} BPM' for bpm in BPMS]
print(classification_report(y_test, y_pred_classes, target_names=label_names))

cm = confusion_matrix(y_test, y_pred_classes)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=label_names, yticklabels=label_names)
plt.title('Confusion Matrix - Test Set', fontweight='bold')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix_fixed.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================
# CELL 11: SAVE MODEL, SCALER, METADATA, TFLITE
# ============================================
model_path = os.path.join(OUTPUT_DIR, 'breath_classifier_logic_fixed.keras')
scaler_path = os.path.join(OUTPUT_DIR, 'scaler_logic_fixed.pkl')
metadata_path = os.path.join(OUTPUT_DIR, 'model_metadata_logic_fixed.json')
tflite_path = os.path.join(OUTPUT_DIR, 'breath_classifier_logic_fixed.tflite')

model.save(model_path)
joblib.dump(scaler, scaler_path)

metadata = {
    'version': 'logic_fixed',
    'task': 'breath_rate_classification',
    'model_mode': MODEL_MODE,
    'sampling_rate_hz': FS,
    'duration_seconds': DURATION,
    'window_size': WINDOW_SIZE,
    'step_size': STEP_SIZE,
    'feature_columns': FEATURE_COLUMNS,
    'num_features': len(FEATURE_COLUMNS),
    'classes_bpm': BPMS,
    'input_shape': [1, WINDOW_SIZE, len(FEATURE_COLUMNS)],
    'output_shape': [1, len(BPMS)],
    'total_params': int(model.count_params()),
    'split_method': 'sample_id_based_no_leakage',
    'train_samples': len(train_ids),
    'val_samples': len(val_ids),
    'test_samples': len(test_ids),
    'train_windows': int(len(X_train)),
    'val_windows': int(len(X_val)),
    'test_windows': int(len(X_test)),
    'best_val_accuracy': float(max(history.history['val_accuracy'])),
    'best_val_loss': float(min(history.history['val_loss'])),
    'test_accuracy': float(test_accuracy),
    'notes': [
        'Data synthetic da them jitter, amplitude modulation, noise va motion burst nhe',
        'Giam overlap window tu 5s step thanh 10s step',
        'Khong augment train truc tiep de bieu do train/val de doc hon',
        'Model nho hon ban high_accuracy_flat cu: Dense(32) + L2 + Dropout',
        'Co kiem tra data leakage train/val/test'
    ]
}
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

# Static batch model for TFLite export
if MODEL_MODE == 'logic_flat_regularized':
    static_model = Sequential([
        Input(batch_shape=(1, WINDOW_SIZE, len(FEATURE_COLUMNS))),
        Flatten(name='flatten_window_features'),
        Dense(
            32,
            activation='relu',
            kernel_regularizer=tf.keras.regularizers.l2(L2_REG),
            name='dense_32_regularized'
        ),
        Dropout(DROPOUT_RATE, name='dropout_regularized'),
        Dense(len(BPMS), activation='softmax', name='output_classifier')
    ], name='breath_logic_fixed_classifier_static')
else:
    static_model = Sequential([
        Input(batch_shape=(1, WINDOW_SIZE, len(FEATURE_COLUMNS))),
        Conv1D(16, kernel_size=11, padding='same', activation='relu', name='conv_1'),
        BatchNormalization(name='bn_1'),
        MaxPooling1D(2, name='pool_1'),
        Conv1D(32, kernel_size=9, padding='same', activation='relu', name='conv_2'),
        BatchNormalization(name='bn_2'),
        GlobalAveragePooling1D(name='global_avg_pool'),
        Dense(48, activation='relu', name='dense_48'),
        Dropout(0.15, name='dropout_1'),
        Dense(len(BPMS), activation='softmax', name='output_classifier')
    ], name='breath_compact_conv_logic_classifier_static')

static_model.set_weights(model.get_weights())
converter = tf.lite.TFLiteConverter.from_keras_model(static_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]
tflite_model = converter.convert()
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

print('Saved model   :', model_path)
print('Saved scaler  :', scaler_path)
print('Saved metadata:', metadata_path)
print(f'Saved TFLite  : {tflite_path} ({len(tflite_model)/1024:.1f} KB)')


In [ ]:
# ============================================
# CELL 12: FINAL SUMMARY + OPTIONAL DOWNLOAD
# ============================================
print('\n' + '=' * 60)
print('FINAL SUMMARY - LOGIC FIXED')
print('=' * 60)
print(f'Model mode       : {MODEL_MODE}')
print(f'Input shape      : [1, {WINDOW_SIZE}, {len(FEATURE_COLUMNS)}]')
print(f'Features         : {FEATURE_COLUMNS}')
print(f'Total params     : {total_params:,}')
print(f'Best val accuracy: {max(history.history["val_accuracy"]) * 100:.2f}%')
print(f'Test accuracy    : {test_accuracy * 100:.2f}%')
print(f'TFLite size      : {len(tflite_model) / 1024:.1f} KB')

print('\nGenerated files:')
for name in sorted(os.listdir(OUTPUT_DIR)):
    path = os.path.join(OUTPUT_DIR, name)
    print(f'  {name:<55} {os.path.getsize(path) / 1024:>8.1f} KB')

try:
    from google.colab import files
    zip_name = 'model_ai_ban_xin_logic_fixed_outputs.zip'
    with zipfile.ZipFile(zip_name, 'w') as zf:
        for name in os.listdir(OUTPUT_DIR):
            zf.write(os.path.join(OUTPUT_DIR, name), arcname=name)
    files.download(zip_name)
except Exception:
    print('Not running in Colab, skip automatic download.')


In [ ]:
# ============================================
# CELL 13: CONVERT KERAS TO TFLITE
# ============================================
# Cell này dùng khi bạn đã có file .keras và muốn convert riêng sang .tflite
# để đẩy model lên cloud. Không cần train lại nếu file .keras đã tồn tại.

from pathlib import Path
import shutil
import tensorflow as tf
from tensorflow.keras.models import load_model, Sequential
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization, Conv1D, MaxPooling1D, GlobalAveragePooling1D, Flatten

# Nếu chạy cell này sau khi restart runtime, các biến dưới đây sẽ tự dùng default.
OUTPUT_DIR = globals().get('OUTPUT_DIR', 'model_ai_ban_xin_logic_fixed_outputs')
WINDOW_SIZE = globals().get('WINDOW_SIZE', 500)
FEATURE_COLUMNS = globals().get(
    'FEATURE_COLUMNS',
    ['acc_mag_filtered', 'gyro_mag', 'spectral_power', 'fft_bpm_norm', 'fft_confidence']
)
BPMS = globals().get('BPMS', [12, 13, 14, 15, 16, 17, 18, 19, 20])
MODEL_MODE = globals().get('MODEL_MODE', 'logic_flat_regularized')
L2_REG = globals().get('L2_REG', 1e-4)
DROPOUT_RATE = globals().get('DROPOUT_RATE', 0.25)

output_dir = Path(OUTPUT_DIR)

keras_candidates = [
    output_dir / 'breath_classifier_logic_fixed.keras',
    output_dir / 'best_breath_classifier.keras',
    output_dir / 'breath_classifier_high_accuracy.keras',
]

KERAS_MODEL_PATH = next((path for path in keras_candidates if path.exists()), None)
if KERAS_MODEL_PATH is None:
    raise FileNotFoundError(
        'Không tìm thấy file .keras trong OUTPUT_DIR. Hãy train model trước hoặc sửa KERAS_MODEL_PATH.\n'
        + '\n'.join(str(path) for path in keras_candidates)
    )

TFLITE_OUTPUT_PATH = output_dir / f'{KERAS_MODEL_PATH.stem}.tflite'

print('Loading Keras model:', KERAS_MODEL_PATH)
loaded_model = load_model(KERAS_MODEL_PATH)

# Tạo static-batch model để TFLite có input cố định [1, 500, 5], hợp với cloud inference.
def build_static_model():
    if MODEL_MODE == 'logic_flat_regularized':
        return Sequential([
            Input(batch_shape=(1, WINDOW_SIZE, len(FEATURE_COLUMNS))),
            Flatten(name='flatten_window_features'),
            Dense(
                32,
                activation='relu',
                kernel_regularizer=tf.keras.regularizers.l2(L2_REG),
                name='dense_32_regularized'
            ),
            Dropout(DROPOUT_RATE, name='dropout_regularized'),
            Dense(len(BPMS), activation='softmax', name='output_classifier')
        ], name='breath_logic_fixed_classifier_static')

    if MODEL_MODE == 'high_accuracy_flat':
        return Sequential([
            Input(batch_shape=(1, WINDOW_SIZE, len(FEATURE_COLUMNS))),
            Flatten(name='flatten_window_features'),
            Dense(64, activation='relu', name='dense_64'),
            Dropout(0.10, name='dropout_1'),
            Dense(len(BPMS), activation='softmax', name='output_classifier')
        ], name='breath_high_accuracy_classifier_static')

    if MODEL_MODE == 'compact_conv':
        return Sequential([
            Input(batch_shape=(1, WINDOW_SIZE, len(FEATURE_COLUMNS))),
            Conv1D(32, kernel_size=9, padding='same', activation='relu', name='conv_1'),
            BatchNormalization(name='bn_1'),
            MaxPooling1D(2, name='pool_1'),
            Dropout(0.10, name='dropout_1'),
            Conv1D(64, kernel_size=7, padding='same', activation='relu', name='conv_2'),
            BatchNormalization(name='bn_2'),
            GlobalAveragePooling1D(name='global_avg_pool'),
            Dense(64, activation='relu', name='dense_64'),
            Dropout(0.10, name='dropout_2'),
            Dense(len(BPMS), activation='softmax', name='output_classifier')
        ], name='breath_compact_conv_classifier_static')

    return None

static_model = build_static_model()
if static_model is not None:
    try:
        static_model.set_weights(loaded_model.get_weights())
        model_for_conversion = static_model
        print('Using static input shape:', [1, WINDOW_SIZE, len(FEATURE_COLUMNS)])
    except Exception as e:
        print('Không set được weights vào static model, fallback convert trực tiếp loaded_model.')
        print('Reason:', e)
        model_for_conversion = loaded_model
else:
    model_for_conversion = loaded_model

converter = tf.lite.TFLiteConverter.from_keras_model(model_for_conversion)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]

tflite_model = converter.convert()
TFLITE_OUTPUT_PATH.write_bytes(tflite_model)
print(f'Saved TFLite: {TFLITE_OUTPUT_PATH} ({len(tflite_model) / 1024:.1f} KB)')

# Tạo folder deploy cho cloud.
deploy_dir = output_dir / 'cloud_deploy'
deploy_dir.mkdir(exist_ok=True)
shutil.copy2(TFLITE_OUTPUT_PATH, deploy_dir / 'breath_v3.tflite')

scaler_candidates = [
    output_dir / 'scaler_logic_fixed.pkl',
    output_dir / 'scaler_high_accuracy.pkl',
]
metadata_candidates = [
    output_dir / 'model_metadata_logic_fixed.json',
    output_dir / 'model_metadata_high_accuracy.json',
]

scaler_path = next((path for path in scaler_candidates if path.exists()), None)
metadata_path = next((path for path in metadata_candidates if path.exists()), None)

if scaler_path is not None:
    shutil.copy2(scaler_path, deploy_dir / 'breath_scaler_v3.joblib')
if metadata_path is not None:
    shutil.copy2(metadata_path, deploy_dir / 'metadata_v3.json')

# Kiểm tra TFLite input/output.
interpreter = tf.lite.Interpreter(model_path=str(TFLITE_OUTPUT_PATH))
interpreter.allocate_tensors()
input_shape = interpreter.get_input_details()[0]['shape']
output_shape = interpreter.get_output_details()[0]['shape']

print('\nTFLite check:')
print('Input shape :', input_shape)
print('Output shape:', output_shape)
print('\nCloud deploy folder:', deploy_dir)
for path in sorted(deploy_dir.iterdir()):
    print(f'  {path.name:<28} {path.stat().st_size / 1024:>8.1f} KB')
